# 🔬 RESNET50 — Experiment 01: Feature Extraction
## SATRIA DATA 2026 — Big Data Challenge
### Smart Waste Classification (Recyclable / Electronic / Organic)

---

Notebook ini adalah **versi monolithic standalone** dari pipeline eksperimen Transfer Learning menggunakan **ResNet50 pretrained ImageNet**.

**Strategi:** Feature Extraction — Backbone dibekukan (frozen), hanya classification head yang dilatih.

**Pipeline:**
1. ⚙️ Setup & Instalasi
2. 📦 Konfigurasi Eksperimen
3. 📂 Dataset & Preprocessing (Splitter + WasteDataset + DataLoader)
4. 🏗️ Arsitektur Model (ResNet50 + Custom Head)
5. 🏋️ Training Engine (Trainer + EarlyStopping)
6. 🚀 Eksekusi Training
7. 📊 Evaluasi & Visualisasi
8. 📝 Kesimpulan & Analisis

> **Cara Pakai di Google Colab:**
> 1. Upload notebook ini ke Colab
> 2. Upload dataset BDC2026 ke Google Drive lalu mount, atau unzip langsung
> 3. Sesuaikan `DATA_ROOT` di Sel 2 (Konfigurasi)
> 4. Jalankan semua sel secara berurutan (Runtime → Run All)

---

---
## ⚙️ Stage 1 — Setup & Instalasi
Menginstal semua dependensi yang diperlukan dan mengimpor library standar.

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES (Colab-ready)
# Jalankan sel ini pertama kali. Aman dijalankan ulang.
# ============================================================
import subprocess, sys

packages = [
    'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121',
    'timm>=1.0.7',
    'torchmetrics>=1.4.0',
    'torchsummary>=1.5.1',
    'albumentations>=1.4.10',
    'scikit-learn>=1.5.0',
    'pandas>=2.2.0',
    'matplotlib>=3.9.0',
    'seaborn>=0.13.2',
    'Pillow>=10.3.0',
    'tqdm>=4.66.0',
    'PyYAML>=6.0.1',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(), check=False)

print('✅ Semua dependensi terinstal.')

In [ ]:
# ============================================================
# STANDARD IMPORTS
# ============================================================
import os
import sys
import json
import csv
import time
import logging
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import transforms
from torchvision.models import resnet50
from torchvision.models.resnet import ResNet50_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('ResNet50_Exp01')

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device   : {DEVICE}')
if torch.cuda.is_available():
    print(f'🎮  GPU      : {torch.cuda.get_device_name(0)}')
    print(f'💾  VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'🐍  Python   : {sys.version.split()[0]}')
print(f'🔥  PyTorch  : {torch.__version__}')

---
## 📦 Stage 2 — Konfigurasi Eksperimen

> ⚠️ **Sesuaikan `DATA_ROOT`** dengan lokasi folder `train/` dataset BDC2026 di environment Anda.
>
> **Struktur folder yang diharapkan:**
> ```
> DATA_ROOT/
> ├── train/
> │   ├── 0_Recyclable/
> │   ├── 1_Electronic/
> │   └── 2_Organic/
> └── test/
> ```

In [ ]:
# ============================================================
# KONFIGURASI EKSPERIMEN (menggantikan resnet50.yaml)
# ============================================================

# --- Sesuaikan path ini dengan environment Anda ---
# Contoh Colab dengan Drive  : '/content/drive/MyDrive/BDC2026'
# Contoh Colab unzip lokal   : '/content/BDC2026'
# Contoh lokal Windows       : r'D:\Satria-Data\data\raw\BDC2026'
DATA_ROOT  = Path('/content/BDC2026')          # ← UBAH INI
TRAIN_DIR  = DATA_ROOT / 'train'
TEST_DIR   = DATA_ROOT / 'test'

# Output dir di Colab
OUTPUT_DIR       = Path('/content/outputs')
CHECKPOINTS_DIR  = OUTPUT_DIR / 'checkpoints'
REPORTS_DIR      = OUTPUT_DIR / 'reports'
FIGURES_DIR      = OUTPUT_DIR / 'figures'

for d in [CHECKPOINTS_DIR, REPORTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Konfigurasi lengkap ----
CONFIG = {
    'experiment': {
        'name'       : 'resnet50_exp01',
        'model'      : 'resnet50',
        'description': 'Transfer Learning ResNet50 — Experiment 01 — Feature Extraction',
        'strategy'   : 'feature_extraction',
        'seed'       : 42,
    },
    'data': {
        'num_classes' : 3,
        'class_names' : ['Recyclable', 'Electronic', 'Organic'],
    },
    'preprocessing': {
        'image_size' : 224,
        'resize_to'  : 256,
        'normalize'  : {
            'mean': [0.485, 0.456, 0.406],
            'std' : [0.229, 0.224, 0.225],
        },
    },
    'augmentation': {
        'random_resized_crop': {'scale': [0.8, 1.0]},
        'horizontal_flip'    : {'p': 0.5},
        'rotation'           : {'degrees': 15},
        'color_jitter'       : {
            'brightness': 0.3, 'contrast': 0.3,
            'saturation': 0.2, 'hue': 0.05,
        },
    },
    'split': {
        'train_ratio': 0.80,
        'val_ratio'  : 0.20,
    },
    'dataloader': {
        'batch_size'          : 32,
        'num_workers'         : 2,    # 2 disarankan di Colab
        'pin_memory'          : True,
        'use_weighted_sampler': True,
    },
    'model': {
        'pretrained'      : True,
        'freeze_backbone' : True,
        'classifier_head' : {'dropout': 0.5},
    },
    'training': {
        'epochs'                  : 20,
        'learning_rate'           : 0.001,
        'weight_decay'            : 0.0001,
        'scheduler_mode'          : 'max',
        'scheduler_factor'        : 0.5,
        'scheduler_patience'      : 3,
        'scheduler_min_lr'        : 1e-5,
        'early_stopping_patience' : 5,
    },
}

# Seed reproducibility
SEED = CONFIG['experiment']['seed']
torch.manual_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = CONFIG['data']['class_names']
EXP_NAME    = CONFIG['experiment']['name']

print('✅ Konfigurasi dimuat.')
print(f'   Experiment : {EXP_NAME}')
print(f'   Strategy   : {CONFIG["experiment"]["strategy"]}')
print(f'   Epochs     : {CONFIG["training"]["epochs"]}')
print(f'   Batch Size : {CONFIG["dataloader"]["batch_size"]}')
print(f'   Seed       : {SEED}')
print(f'   Output Dir : {OUTPUT_DIR}')

---
## 📂 Stage 3 — Dataset & Preprocessing

### Komponen:
- **`LABEL_MAP`** — Pemetaan nama folder ke integer label
- **`collect_dataset()`** — Scan folder train, kumpulkan semua path gambar
- **`stratified_split()`** — Stratified 80/20 split menjaga distribusi kelas
- **`WasteDataset`** — Custom PyTorch Dataset dengan graceful error handling
- **`get_train_transforms()` / `get_val_transforms()`** — Pipeline augmentasi & normalisasi
- **`build_train_loader()` / `build_val_loader()`** — DataLoader dengan WeightedRandomSampler

In [ ]:
# ============================================================
# 3A. SPLITTER — collect_dataset & stratified_split
# (dari src/preprocessing/splitter.py)
# ============================================================

LABEL_MAP: Dict[str, int] = {
    '0_Recyclable': 0,
    '1_Electronic' : 1,
    '2_Organic'    : 2,
}

INT_TO_CLASS: Dict[int, str] = {
    0: 'Recyclable',
    1: 'Electronic',
    2: 'Organic',
}

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}


def collect_dataset(train_dir: Path) -> pd.DataFrame:
    """
    Mengumpulkan seluruh path gambar dan label dari direktori train.

    Struktur folder yang diharapkan:
        train_dir/
        ├── 0_Recyclable/
        ├── 1_Electronic/
        └── 2_Organic/

    Returns:
        pd.DataFrame dengan kolom ['filepath', 'label', 'class_name', 'folder']
    """
    if not train_dir.exists():
        raise FileNotFoundError(f'Train directory tidak ditemukan: {train_dir}')

    records = []
    detected_classes = []

    for class_folder in sorted(train_dir.iterdir()):
        if not class_folder.is_dir():
            continue
        folder_name = class_folder.name
        if folder_name not in LABEL_MAP:
            logger.warning(f'Folder tidak dikenal, dilewati: {folder_name}')
            continue

        label      = LABEL_MAP[folder_name]
        class_name = INT_TO_CLASS[label]
        detected_classes.append(folder_name)

        for img_path in class_folder.iterdir():
            if img_path.is_file() and img_path.suffix.lower() in VALID_EXTENSIONS:
                records.append({
                    'filepath'  : str(img_path),
                    'label'     : label,
                    'class_name': class_name,
                    'folder'    : folder_name,
                })

    if not records:
        raise ValueError(f'Tidak ditemukan gambar di: {train_dir}')

    logger.info(f'Kelas terdeteksi: {detected_classes}')
    logger.info(f'Total gambar dikumpulkan: {len(records):,}')
    return pd.DataFrame(records)


def stratified_split(
    df: pd.DataFrame,
    train_ratio: float = 0.80,
    val_ratio: float = 0.20,
    random_seed: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Membagi DataFrame menjadi train dan validation set secara stratified.
    Memastikan proporsi kelas yang seimbang di setiap split.
    """
    total = train_ratio + val_ratio
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f'train_ratio + val_ratio harus = 1.0. Diterima: {total:.4f}')

    df_train, df_val = train_test_split(
        df,
        test_size=val_ratio,
        stratify=df['label'],
        random_state=random_seed,
        shuffle=True,
    )
    df_train = df_train.reset_index(drop=True)
    df_val   = df_val.reset_index(drop=True)

    logger.info(f'Split selesai (seed={random_seed}):')
    logger.info(f'  Train : {len(df_train):,} gambar ({train_ratio*100:.0f}%)')
    logger.info(f'  Val   : {len(df_val):,} gambar ({val_ratio*100:.0f}%)')
    return df_train, df_val


def get_split_summary(df_train: pd.DataFrame, df_val: pd.DataFrame) -> str:
    """Menghasilkan teks ringkasan split untuk ditampilkan di notebook."""
    total = len(df_train) + len(df_val)
    lines = [
        '=' * 52,
        '  DATASET SPLIT SUMMARY',
        '=' * 52,
        f'  Total gambar      : {total:,}',
        f'  Training set      : {len(df_train):,} ({len(df_train)/total*100:.1f}%)',
        f'  Validation set    : {len(df_val):,} ({len(df_val)/total*100:.1f}%)',
        '',
        '  Training Distribution:',
    ]
    for cls, cnt in df_train['class_name'].value_counts().items():
        pct = cnt / len(df_train) * 100
        lines.append(f'    {cls:<14}: {cnt:,} ({pct:.1f}%)')
    lines.append('  Validation Distribution:')
    for cls, cnt in df_val['class_name'].value_counts().items():
        pct = cnt / len(df_val) * 100
        lines.append(f'    {cls:<14}: {cnt:,} ({pct:.1f}%)')
    lines.append('=' * 52)
    return '\n'.join(lines)


print('✅ Fungsi Splitter didefinisikan.')

In [ ]:
# ============================================================
# 3B. TRANSFORMS — Pipeline Augmentasi & Normalisasi
# (dari src/preprocessing/transforms.py)
# ============================================================

def get_train_transforms(
    image_size: int = 224,
    resize_to: int = 256,
    mean: List[float] = [0.485, 0.456, 0.406],
    std:  List[float] = [0.229, 0.224, 0.225],
    rotation_degrees: int = 15,
    jitter_brightness: float = 0.3,
    jitter_contrast: float = 0.3,
    jitter_saturation: float = 0.2,
    jitter_hue: float = 0.05,
    flip_p: float = 0.5,
    crop_scale: Tuple[float, float] = (0.8, 1.0),
) -> transforms.Compose:
    """
    Pipeline transformasi untuk Training Set.
    Urutan: RandomResizedCrop → HorizontalFlip → Rotation → ColorJitter → ToTensor → Normalize
    """
    return transforms.Compose([
        transforms.RandomResizedCrop(size=image_size, scale=crop_scale),
        transforms.RandomHorizontalFlip(p=flip_p),
        transforms.RandomRotation(degrees=rotation_degrees),
        transforms.ColorJitter(
            brightness=jitter_brightness,
            contrast=jitter_contrast,
            saturation=jitter_saturation,
            hue=jitter_hue,
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])


def get_val_transforms(
    image_size: int = 224,
    resize_to: int = 256,
    mean: List[float] = [0.485, 0.456, 0.406],
    std:  List[float] = [0.229, 0.224, 0.225],
) -> transforms.Compose:
    """
    Pipeline transformasi deterministik untuk Validation/Test Set.
    Urutan: Resize(256) → CenterCrop(224) → ToTensor → Normalize
    """
    return transforms.Compose([
        transforms.Resize(resize_to),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])


print('✅ Fungsi Transforms didefinisikan.')

In [ ]:
# ============================================================
# 3C. WASTE DATASET — Custom PyTorch Dataset
# (dari src/datasets/waste_dataset.py)
# ============================================================

class WasteDataset(Dataset):
    """
    Custom PyTorch Dataset untuk dataset klasifikasi sampah BDC 2026.

    Menerima DataFrame dari stratified_split() dengan kolom:
      - 'filepath'   : Path lengkap ke file gambar
      - 'label'      : Integer label (0=Recyclable, 1=Electronic, 2=Organic)
      - 'class_name' : Nama kelas

    Fitur:
      - Graceful fallback untuk gambar corrupt (tidak crash saat training)
      - WeightedRandomSampler support via get_sample_weights()
    """

    CLASS_NAMES = ['Recyclable', 'Electronic', 'Organic']

    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform: Optional[Callable] = None,
        split_name: str = 'unknown',
    ) -> None:
        self.df         = dataframe.reset_index(drop=True)
        self.transform  = transform
        self.split_name = split_name
        self.class_names = self.CLASS_NAMES

        required_cols = {'filepath', 'label'}
        missing = required_cols - set(self.df.columns)
        if missing:
            raise ValueError(f'DataFrame kurang kolom: {missing}')

        logger.info(
            f'WasteDataset [{split_name}] dibuat: '
            f'{len(self.df):,} gambar'
        )

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        row      = self.df.iloc[idx]
        filepath = Path(row['filepath'])
        label    = int(row['label'])

        try:
            image = Image.open(filepath).convert('RGB')
        except (UnidentifiedImageError, OSError, Exception) as e:
            logger.warning(f'Gambar tidak bisa dibaca: {filepath.name} — {e}')
            fallback_idx = (idx + 1) % len(self.df)
            return self.__getitem__(fallback_idx)

        if self.transform is not None:
            image = self.transform(image)

        return image, label

    def get_class_counts(self) -> dict:
        return self.df['class_name'].value_counts().to_dict()

    def get_class_weights(self) -> torch.Tensor:
        counts  = self.df['label'].value_counts().sort_index()
        total   = len(self.df)
        n_cls   = len(self.CLASS_NAMES)
        weights = total / (n_cls * counts.values)
        return torch.tensor(weights, dtype=torch.float32)

    def get_sample_weights(self) -> torch.Tensor:
        class_weights  = self.get_class_weights()
        sample_weights = torch.tensor(
            [float(class_weights[label]) for label in self.df['label']],
            dtype=torch.float32
        )
        return sample_weights

    def __repr__(self) -> str:
        counts  = self.get_class_counts()
        dist_str = ', '.join(f'{k}:{v}' for k, v in counts.items())
        return f"WasteDataset(split='{self.split_name}', n={len(self)}, [{dist_str}])"


print('✅ WasteDataset didefinisikan.')

In [ ]:
# ============================================================
# 3D. DATALOADER — Factory Functions
# (dari src/preprocessing/dataloader.py)
# ============================================================

def build_train_loader(
    df_train: pd.DataFrame,
    batch_size: int = 32,
    num_workers: int = 2,
    pin_memory: bool = True,
    image_size: int = 224,
    resize_to: int = 256,
    mean: list = [0.485, 0.456, 0.406],
    std: list  = [0.229, 0.224, 0.225],
    use_weighted_sampler: bool = True,
    seed: int = 42,
) -> DataLoader:
    """
    Membangun DataLoader untuk Training Set.
    Menggunakan WeightedRandomSampler untuk menangani class imbalance.
    """
    transform = get_train_transforms(
        image_size=image_size, resize_to=resize_to, mean=mean, std=std,
    )
    dataset = WasteDataset(dataframe=df_train, transform=transform, split_name='train')

    if use_weighted_sampler:
        sample_weights = dataset.get_sample_weights()
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
            generator=torch.Generator().manual_seed(seed),
        )
        shuffle = False
        logger.info('WeightedRandomSampler diaktifkan untuk training.')
    else:
        sampler = None
        shuffle = True

    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, sampler=sampler,
        num_workers=num_workers, pin_memory=pin_memory, drop_last=True,
    )
    logger.info(f'Train DataLoader: {len(dataset):,} gambar, {len(loader)} batches')
    return loader


def build_val_loader(
    df_val: pd.DataFrame,
    batch_size: int = 32,
    num_workers: int = 2,
    pin_memory: bool = True,
    image_size: int = 224,
    resize_to: int = 256,
    mean: list = [0.485, 0.456, 0.406],
    std: list  = [0.229, 0.224, 0.225],
) -> DataLoader:
    """
    Membangun DataLoader untuk Validation Set.
    Deterministik — tidak ada augmentasi, tidak ada sampler khusus.
    """
    transform = get_val_transforms(
        image_size=image_size, resize_to=resize_to, mean=mean, std=std,
    )
    dataset = WasteDataset(dataframe=df_val, transform=transform, split_name='val')
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory, drop_last=False,
    )
    logger.info(f'Val DataLoader: {len(dataset):,} gambar, {len(loader)} batches')
    return loader


print('✅ Fungsi DataLoader didefinisikan.')

In [ ]:
# ============================================================
# EKSEKUSI PREPROCESSING
# ============================================================
pre_cfg = CONFIG['preprocessing']
dl_cfg  = CONFIG['dataloader']
spl_cfg = CONFIG['split']

# 1. Kumpulkan dataset
print('📂 Mengumpulkan dataset...')
df_all = collect_dataset(TRAIN_DIR)

# 2. Stratified Split
print('✂️  Melakukan stratified split...')
df_train, df_val = stratified_split(
    df=df_all,
    train_ratio=spl_cfg['train_ratio'],
    val_ratio=spl_cfg['val_ratio'],
    random_seed=SEED,
)
print(get_split_summary(df_train, df_val))

# Simpan split report
df_train.assign(split='train').pipe(
    lambda d: pd.concat([d, df_val.assign(split='val')], ignore_index=True)
).to_csv(REPORTS_DIR / 'preprocessing_split.csv', index=False)
print(f'📄 Split report disimpan: {REPORTS_DIR / "preprocessing_split.csv"}')

# 3. Bangun DataLoader
print('\n🔄 Membangun DataLoader...')
train_loader = build_train_loader(
    df_train=df_train,
    batch_size=dl_cfg['batch_size'],
    num_workers=dl_cfg['num_workers'],
    pin_memory=dl_cfg['pin_memory'],
    image_size=pre_cfg['image_size'],
    resize_to=pre_cfg['resize_to'],
    mean=pre_cfg['normalize']['mean'],
    std=pre_cfg['normalize']['std'],
    use_weighted_sampler=dl_cfg['use_weighted_sampler'],
    seed=SEED,
)

val_loader = build_val_loader(
    df_val=df_val,
    batch_size=dl_cfg['batch_size'],
    num_workers=dl_cfg['num_workers'],
    pin_memory=dl_cfg['pin_memory'],
    image_size=pre_cfg['image_size'],
    resize_to=pre_cfg['resize_to'],
    mean=pre_cfg['normalize']['mean'],
    std=pre_cfg['normalize']['std'],
)

# 4. Verifikasi satu batch
sample_images, sample_labels = next(iter(train_loader))
print(f'\n📊 Verifikasi batch pertama:')
print(f'   Image shape  : {sample_images.shape}')
print(f'   Labels       : {sample_labels.tolist()[:8]}...')
print(f'   Pixel range  : [{sample_images.min():.3f}, {sample_images.max():.3f}]')
print(f'   Unique labels: {sorted(sample_labels.unique().tolist())}')
print(f'\n✅ Preprocessing selesai!')
print(f'   Train batches: {len(train_loader)}')
print(f'   Val batches  : {len(val_loader)}')

---
## 🏗️ Stage 4 — Arsitektur Model ResNet50

### Desain Arsitektur:
```
Input (224×224×3)
        │
  ┌─────▼──────────────────────────────────────────────────┐
  │  ResNet50 Backbone (FROZEN — pretrained ImageNet)       │
  │  Conv1 → BN → ReLU → MaxPool                           │
  │  Layer1 (3 bottleneck blocks)                          │
  │  Layer2 (4 bottleneck blocks)                          │
  │  Layer3 (6 bottleneck blocks)                          │
  │  Layer4 (3 bottleneck blocks)                          │
  │  AdaptiveAvgPool → Flatten                             │
  └─────────────────────────────────────┬──────────────────┘
                                        │ 2048-dim features
  ┌─────────────────────────────────────▼──────────────────┐
  │  Custom Head (TRAINABLE)                               │
  │  Dropout(0.5) → Linear(2048 → 3)                      │
  └─────────────────────────────────────────────────────────┘
                                        │
                                 Logits (3 kelas)
```

In [ ]:
# ============================================================
# 4. MODEL — ResNet50WasteClassifier
# (dari src/models/transfer_learning/resnet50.py)
# ============================================================

class ResNet50WasteClassifier(nn.Module):
    """
    Model klasifikasi sampah berbasis ResNet50 Pretrained.

    Arsitektur:
      - Backbone : ResNet50 pretrained ImageNet (frozen untuk Feature Extraction)
      - Head     : Dropout(p) + Linear(2048 → num_classes)

    Args:
        num_classes (int): Jumlah kelas output (default: 3)
        pretrained  (bool): Menggunakan bobot ImageNet (default: True)
        dropout_p   (float): Probabilitas dropout pada classification head
    """

    def __init__(
        self,
        num_classes: int = 3,
        pretrained: bool = True,
        dropout_p: float = 0.5,
    ) -> None:
        super().__init__()

        self.num_classes = num_classes
        self.pretrained  = pretrained
        self.dropout_p   = dropout_p

        # Load backbone
        weights        = ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone  = resnet50(weights=weights)
        in_features    = self.backbone.fc.in_features

        # Ganti classification head
        if dropout_p > 0:
            self.backbone.fc = nn.Sequential(
                nn.Dropout(p=dropout_p),
                nn.Linear(in_features, num_classes)
            )
        else:
            self.backbone.fc = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: (B, 3, 224, 224) → logits (B, num_classes)"""
        return self.backbone(x)

    def freeze_backbone(self) -> None:
        """Freeze seluruh backbone, hanya head yang trainable (Feature Extraction)."""
        for param in self.backbone.parameters():
            param.requires_grad = False
        for param in self.backbone.fc.parameters():
            param.requires_grad = True

    def unfreeze_all(self) -> None:
        """Unfreeze seluruh model (untuk Fine-Tuning)."""
        for param in self.parameters():
            param.requires_grad = True

    def get_trainable_params_count(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_total_params_count(self) -> int:
        return sum(p.numel() for p in self.parameters())


def build_resnet50(config: Dict[str, Any]) -> ResNet50WasteClassifier:
    """
    Factory function untuk membangun model ResNet50 dari dictionary konfigurasi.
    """
    model_cfg   = config.get('model', {})
    num_classes = config.get('data', {}).get('num_classes', 3)
    pretrained  = model_cfg.get('pretrained', True)
    dropout_p   = model_cfg.get('classifier_head', {}).get('dropout', 0.5)

    model = ResNet50WasteClassifier(
        num_classes=num_classes, pretrained=pretrained, dropout_p=dropout_p
    )
    if model_cfg.get('freeze_backbone', True):
        model.freeze_backbone()

    return model


def get_backbone_info(model: ResNet50WasteClassifier) -> Dict[str, Any]:
    """Mengekstrak informasi backbone untuk pelaporan."""
    total     = model.get_total_params_count()
    trainable = model.get_trainable_params_count()
    frozen    = total - trainable
    return {
        'architecture'      : 'ResNet50',
        'pretrained_source' : 'torchvision.models',
        'pretrained_dataset': 'ImageNet-1K',
        'input_resolution'  : '224x224',
        'total_parameters'  : total,
        'trainable_parameters': trainable,
        'frozen_parameters' : frozen,
        'classification_head': 'Dropout + Linear' if model.dropout_p > 0 else 'Linear',
        'strategy'          : 'Feature Extraction' if trainable < 100_000 else 'Fine-Tuning',
        'bdc_compliant'     : True,
        'note'              : 'Backbone tidak pernah dilatih dengan data kompetisi BDC',
    }


# Inisialisasi model
print('🏗️  Membangun model ResNet50...')
model = build_resnet50(CONFIG)

print('\n📋 Backbone Info:')
info = get_backbone_info(model)
for k, v in info.items():
    if isinstance(v, int):
        print(f'   {k:<25}: {v:,}')
    else:
        print(f'   {k:<25}: {v}')

print(f'\n✅ Model siap: {info["trainable_parameters"]:,} parameter trainable dari {info["total_parameters"]:,} total')

---
## 🏋️ Stage 5 — Training Engine

### Komponen:
- **`EarlyStopping`** — Monitor val_macro_f1, simpan best weights otomatis
- **`train_one_epoch()`** — Satu epoch training dengan gradient update
- **`validate_one_epoch()`** — Satu epoch evaluasi tanpa gradient
- **`Trainer`** — Orkestrasi lengkap: loop training, scheduler, checkpoint, history CSV

In [ ]:
# ============================================================
# 5A. EARLY STOPPING
# (dari src/training/early_stopping.py)
# ============================================================

class EarlyStopping:
    """
    Early Stopping berbasis val_macro_f1 (higher is better).

    Menghentikan training jika val_macro_f1 tidak meningkat
    dalam `patience` epoch berturut-turut.
    Menyimpan best model weights secara otomatis.

    Args:
        patience  : Epoch tanpa peningkatan sebelum stop (default 5)
        min_delta : Peningkatan minimum yang dianggap signifikan (default 1e-4)
        verbose   : Tampilkan log (default True)
    """

    def __init__(self, patience: int = 5, min_delta: float = 1e-4, verbose: bool = True) -> None:
        self.patience   = patience
        self.min_delta  = min_delta
        self.verbose    = verbose

        self.best_score   : Optional[float] = None
        self.best_weights : Optional[dict]  = None
        self.best_epoch   : int  = 0
        self.counter      : int  = 0
        self.early_stop   : bool = False

    def __call__(self, score: float, model: nn.Module, epoch: int) -> None:
        if self.best_score is None:
            self._update_best(score, model, epoch)
        elif score >= self.best_score + self.min_delta:
            if self.verbose:
                logger.info(f'[EarlyStopping] Val F1 meningkat: {self.best_score:.4f} → {score:.4f} ✓')
            self._update_best(score, model, epoch)
        else:
            self.counter += 1
            if self.verbose:
                logger.info(f'[EarlyStopping] Tidak ada peningkatan. Counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                logger.info(f'[EarlyStopping] Training dihentikan di epoch {epoch}. Best epoch: {self.best_epoch} (F1={self.best_score:.4f})')

    def _update_best(self, score: float, model: nn.Module, epoch: int) -> None:
        self.best_score   = score
        self.best_epoch   = epoch
        self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        self.counter      = 0

    def get_summary(self) -> dict:
        return {
            'best_epoch'    : self.best_epoch,
            'best_val_f1'   : round(self.best_score, 4) if self.best_score else None,
            'stopped_early' : self.early_stop,
            'patience'      : self.patience,
            'final_counter' : self.counter,
        }


print('✅ EarlyStopping didefinisikan.')

In [ ]:
# ============================================================
# 5B. TRAINER — Training Engine
# (dari src/training/trainer.py)
# ============================================================

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epoch: int,
) -> Dict[str, float]:
    """Menjalankan satu epoch training. Returns: {'train_loss', 'train_acc', 'train_macro_f1'}"""
    model.train()
    running_loss = 0.0
    all_preds    = []
    all_labels   = []

    pbar = tqdm(loader, desc=f'Train Ep{epoch:02d}', leave=False, dynamic_ncols=True)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = running_loss / len(loader)
    return {
        'train_loss'     : round(avg_loss, 6),
        'train_acc'      : round(accuracy_score(all_labels, all_preds), 6),
        'train_macro_f1' : round(f1_score(all_labels, all_preds, average='macro', zero_division=0), 6),
    }


def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    epoch: int,
) -> Dict[str, float]:
    """Menjalankan satu epoch validasi. Returns: {'val_loss', 'val_acc', 'val_macro_f1', ...}"""
    model.eval()
    running_loss = 0.0
    all_preds    = []
    all_labels   = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f'Val   Ep{epoch:02d}', leave=False, dynamic_ncols=True):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            loss   = criterion(logits, labels)
            running_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader)
    return {
        'val_loss'      : round(avg_loss, 6),
        'val_acc'       : round(accuracy_score(all_labels, all_preds), 6),
        'val_macro_f1'  : round(f1_score(all_labels, all_preds, average='macro', zero_division=0), 6),
        'val_precision' : round(precision_score(all_labels, all_preds, average='macro', zero_division=0), 6),
        'val_recall'    : round(recall_score(all_labels, all_preds, average='macro', zero_division=0), 6),
    }


class Trainer:
    """
    Training orchestrator untuk Smart Waste Classification.

    Mengintegrasikan:
      - train_one_epoch() + validate_one_epoch()
      - EarlyStopping (berbasis val_macro_f1)
      - ReduceLROnPlateau scheduler
      - Checkpoint management (best & last model)
      - Training history (CSV)

    Cara pakai:
        trainer = Trainer(model=model, config=CONFIG)
        history = trainer.train(train_loader, val_loader)
    """

    def __init__(
        self,
        model: nn.Module,
        config: dict,
        checkpoints_dir: Path = CHECKPOINTS_DIR,
        reports_dir: Path = REPORTS_DIR,
    ) -> None:
        self.device = DEVICE
        self.model  = model.to(self.device)
        self.config = config
        self.checkpoints_dir = checkpoints_dir
        self.reports_dir     = reports_dir

        train_cfg = config['training']
        exp_cfg   = config['experiment']

        self.optimizer = AdamW(
            model.parameters(),
            lr=train_cfg['learning_rate'],
            weight_decay=train_cfg['weight_decay'],
        )
        self.criterion = nn.CrossEntropyLoss()
        self.scheduler = ReduceLROnPlateau(
            self.optimizer,
            mode=train_cfg['scheduler_mode'],
            factor=train_cfg['scheduler_factor'],
            patience=train_cfg['scheduler_patience'],
            min_lr=train_cfg['scheduler_min_lr'],
        )
        self.early_stopping = EarlyStopping(
            patience=train_cfg['early_stopping_patience'],
            verbose=True,
        )
        self.epochs   = train_cfg['epochs']
        self.exp_name = exp_cfg['name']
        self.history  : List[Dict] = []

    def train(self, train_loader: DataLoader, val_loader: DataLoader) -> List[Dict]:
        """Menjalankan full training loop. Returns: List training history per epoch."""
        logger.info('=' * 60)
        logger.info(f'  TRAINING DIMULAI: {self.exp_name}')
        logger.info(f'  Device   : {self.device}')
        logger.info(f'  Epochs   : {self.epochs}')
        logger.info(f'  LR       : {self.config["training"]["learning_rate"]}')
        logger.info(f'  ES Patience: {self.config["training"]["early_stopping_patience"]}')
        logger.info('=' * 60)

        total_start = time.time()

        for epoch in range(1, self.epochs + 1):
            epoch_start  = time.time()
            train_metrics = train_one_epoch(self.model, train_loader, self.criterion, self.optimizer, self.device, epoch)
            val_metrics   = validate_one_epoch(self.model, val_loader, self.criterion, self.device, epoch)
            epoch_time    = time.time() - epoch_start
            current_lr    = self.optimizer.param_groups[0]['lr']

            epoch_log = {
                'epoch'        : epoch,
                'lr'           : current_lr,
                'epoch_time_s' : round(epoch_time, 2),
                **train_metrics,
                **val_metrics,
            }
            self.history.append(epoch_log)

            logger.info(
                f'Epoch [{epoch:02d}/{self.epochs}] '
                f'train_loss={train_metrics["train_loss"]:.4f} '
                f'train_f1={train_metrics["train_macro_f1"]:.4f} | '
                f'val_loss={val_metrics["val_loss"]:.4f} '
                f'val_f1={val_metrics["val_macro_f1"]:.4f} '
                f'lr={current_lr:.6f} ({epoch_time:.1f}s)'
            )

            self.scheduler.step(val_metrics['val_macro_f1'])
            self.early_stopping(score=val_metrics['val_macro_f1'], model=self.model, epoch=epoch)
            self._save_checkpoint(epoch, tag='last')

            if self.early_stopping.early_stop:
                logger.info(f'Early stopping triggered di epoch {epoch}.')
                break

        total_time = time.time() - total_start

        # Restore best weights
        if self.early_stopping.best_weights is not None:
            self.model.load_state_dict(self.early_stopping.best_weights)
            logger.info(f'Best weights restored dari epoch {self.early_stopping.best_epoch} (F1={self.early_stopping.best_score:.4f})')

        self._save_checkpoint(self.early_stopping.best_epoch, tag='best')
        self._save_history()

        es_summary = self.early_stopping.get_summary()
        summary = {
            'experiment'           : self.exp_name,
            'total_epochs_trained' : len(self.history),
            'total_time_min'       : round(total_time / 60, 2),
            'best_epoch'           : es_summary['best_epoch'],
            'best_val_macro_f1'    : es_summary['best_val_f1'],
            'stopped_early'        : es_summary['stopped_early'],
            'final_train_loss'     : self.history[-1]['train_loss'],
            'final_val_loss'       : self.history[-1]['val_loss'],
            'final_val_f1'         : self.history[-1]['val_macro_f1'],
        }
        self._save_summary(summary)

        logger.info('=' * 60)
        logger.info('  TRAINING SELESAI')
        logger.info(f'  Best Epoch    : {summary["best_epoch"]}')
        logger.info(f'  Best Val F1   : {summary["best_val_macro_f1"]:.4f}')
        logger.info(f'  Total Time    : {summary["total_time_min"]:.1f} menit')
        logger.info('=' * 60)

        return self.history

    def _save_checkpoint(self, epoch: int, tag: str = 'last') -> None:
        self.checkpoints_dir.mkdir(parents=True, exist_ok=True)
        save_path = self.checkpoints_dir / f'{self.exp_name}_{tag}_model.pth'
        torch.save({
            'epoch'            : epoch,
            'model_state'      : self.model.state_dict(),
            'optimizer_state'  : self.optimizer.state_dict(),
            'config'           : self.config,
            'tag'              : tag,
        }, save_path)
        logger.debug(f'Checkpoint saved: {save_path}')

    def _save_history(self) -> None:
        self.reports_dir.mkdir(parents=True, exist_ok=True)
        csv_path = self.reports_dir / f'{self.exp_name}_training_history.csv'
        if self.history:
            fieldnames = list(self.history[0].keys())
            with open(csv_path, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(self.history)
        logger.info(f'History saved: {csv_path}')

    def _save_summary(self, summary: dict) -> None:
        self.reports_dir.mkdir(parents=True, exist_ok=True)
        json_path = self.reports_dir / f'{self.exp_name}_experiment_summary.json'
        with open(json_path, 'w') as f:
            json.dump(summary, f, indent=2)
        logger.info(f'Summary saved: {json_path}')


print('✅ Training Engine (EarlyStopping + Trainer) didefinisikan.')

---
## 🚀 Stage 6 — Eksekusi Training

> 💡 Sel ini menjalankan training penuh. Estimasi waktu:
> - Colab T4 GPU: ~3–5 menit/epoch → ±1 jam (20 epoch)
> - Colab A100 GPU: ~1–2 menit/epoch → ±30 menit
> 
> Early stopping akan menghentikan training lebih awal jika model konvergen.

In [ ]:
# ============================================================
# EKSEKUSI TRAINING
# ============================================================
print('🚀 Memulai training...')
print(f'   Model     : ResNet50 Feature Extraction')
print(f'   Device    : {DEVICE}')
print(f'   Epochs    : {CONFIG["training"]["epochs"]}')
print(f'   LR        : {CONFIG["training"]["learning_rate"]}')
print()

trainer = Trainer(
    model=model,
    config=CONFIG,
    checkpoints_dir=CHECKPOINTS_DIR,
    reports_dir=REPORTS_DIR,
)

history = trainer.train(train_loader, val_loader)
df_history = pd.DataFrame(history)

print('\n📊 Training History (5 Epoch Terakhir):')
display(df_history[['epoch', 'lr', 'train_loss', 'train_macro_f1', 'val_loss', 'val_macro_f1']].tail())

---
## 📊 Stage 7 — Evaluasi & Visualisasi

### Komponen:
- **`run_inference()`** — Forward pass seluruh validation set
- **`compute_metrics()`** — Hitung Macro F1, Accuracy, Precision, Recall per kelas
- **`plot_confusion_matrix()`** — Heatmap confusion matrix (Count + Normalized)
- **`plot_learning_curves()`** — Kurva Loss, Accuracy, Macro F1
- **`plot_per_class_metrics()`** — Bar chart Precision/Recall/F1 per kelas
- **`plot_error_analysis()`** — Grid gambar salah prediksi

In [ ]:
# ============================================================
# 7A. EVALUATION FUNCTIONS
# (dari src/evaluation/metrics.py + visualizer.py)
# ============================================================

# --- Warna palet (konsisten) ---
DARK_BG    = '#1e1e2e'
PANEL_BG   = '#313244'
TEXT_COLOR = '#cdd6f4'
GRID_COLOR = '#45475a'
COLORS     = ['#89dceb', '#a6e3a1', '#fab387']
ACCENT     = '#f38ba8'


def _base_style(ax):
    ax.set_facecolor(DARK_BG)
    ax.tick_params(colors=TEXT_COLOR)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_color(GRID_COLOR)
    ax.grid(alpha=0.3, color=GRID_COLOR)


def run_inference(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Inferensi seluruh DataLoader. Returns: (y_pred, y_true, y_probs)"""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Inferensi', leave=False):
            images = images.to(device, non_blocking=True)
            logits = model(images)
            probs  = torch.softmax(logits, dim=1)
            preds  = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str] = CLASS_NAMES) -> Dict:
    """Menghitung Macro F1, Accuracy, Precision, Recall, dan metrik per kelas."""
    macro_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    f1_per    = f1_score(y_true, y_pred, average=None, zero_division=0)
    prec_per  = precision_score(y_true, y_pred, average=None, zero_division=0)
    rec_per   = recall_score(y_true, y_pred, average=None, zero_division=0)

    per_class = {}
    for i, cls in enumerate(class_names):
        per_class[cls] = {
            'precision': round(float(prec_per[i]), 4),
            'recall'   : round(float(rec_per[i]), 4),
            'f1'       : round(float(f1_per[i]), 4),
            'support'  : int(np.sum(y_true == i)),
        }
    return {
        'macro_f1'  : round(float(macro_f1), 4),
        'accuracy'  : round(float(accuracy), 4),
        'precision' : round(float(precision), 4),
        'recall'    : round(float(recall), 4),
        'per_class' : per_class,
    }


def get_classification_report(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str] = CLASS_NAMES) -> str:
    return classification_report(y_true, y_pred, target_names=class_names, zero_division=0, digits=4)


def get_misclassified_samples(
    y_true: np.ndarray, y_pred: np.ndarray,
    filepaths: List[str], class_names: List[str] = CLASS_NAMES, max_samples: int = 12,
) -> List[Dict]:
    wrong = np.where(y_true != y_pred)[0]
    return [{
        'filepath'  : filepaths[i],
        'true_label': int(y_true[i]), 'pred_label': int(y_pred[i]),
        'true_class': class_names[y_true[i]], 'pred_class': class_names[y_pred[i]],
    } for i in wrong[:max_samples]]


def plot_confusion_matrix(cm: np.ndarray, class_names=CLASS_NAMES, save_path=None, experiment_name='ResNet50 (Exp 01)'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle(f'Confusion Matrix — {experiment_name}', fontsize=15, fontweight='bold', color=TEXT_COLOR)

    for ax, (data, fmt, title) in zip(axes, [
        (cm, 'd', 'Count'),
        (cm.astype(float) / cm.sum(axis=1, keepdims=True), '.3f', 'Normalized (Row = Recall)'),
    ]):
        ax.set_facecolor(DARK_BG)
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.5, linecolor=DARK_BG, annot_kws={'size': 11, 'weight': 'bold'})
        ax.set_title(title, fontsize=12, color=TEXT_COLOR, pad=10)
        ax.set_xlabel('Predicted Label', color=TEXT_COLOR)
        ax.set_ylabel('True Label', color=TEXT_COLOR)
        ax.tick_params(colors=TEXT_COLOR, labelsize=10)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
        logger.info(f'Saved: {save_path}')
    plt.show()


def plot_learning_curves(history_df: pd.DataFrame, best_epoch: int, save_path=None, experiment_name='ResNet50 (Exp 01)'):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle(f'Learning Curves — {experiment_name}', fontsize=15, fontweight='bold', color=TEXT_COLOR)

    pairs = [
        ('train_loss',     'val_loss',     'Loss',     '#f38ba8', '#89dceb'),
        ('train_acc',      'val_acc',      'Accuracy', '#a6e3a1', '#fab387'),
        ('train_macro_f1', 'val_macro_f1', 'Macro F1', '#cba6f7', '#f9e2af'),
    ]
    epochs = history_df['epoch'].tolist()
    for ax, (tc, vc, title, c1, c2) in zip(axes, pairs):
        _base_style(ax)
        ax.plot(epochs, history_df[tc], color=c1, label='Train', linewidth=2, alpha=0.9)
        ax.plot(epochs, history_df[vc], color=c2, label='Val', linewidth=2, linestyle='--', alpha=0.9)
        ax.axvline(x=best_epoch, color='#f9e2af', linestyle=':', linewidth=1.5, alpha=0.8, label=f'Best Ep.{best_epoch}')
        ax.set_title(title, fontsize=13, color=TEXT_COLOR)
        ax.set_xlabel('Epoch', fontsize=11)
        ax.legend(fontsize=9, facecolor=PANEL_BG, labelcolor=TEXT_COLOR, framealpha=0.8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
        logger.info(f'Saved: {save_path}')
    plt.show()


def plot_per_class_metrics(metrics: Dict, class_names=CLASS_NAMES, save_path=None, experiment_name='ResNet50 (Exp 01)'):
    per_class = metrics.get('per_class', {})
    prec = [per_class[c]['precision'] for c in class_names]
    rec  = [per_class[c]['recall']    for c in class_names]
    f1   = [per_class[c]['f1']        for c in class_names]

    x, width = np.arange(len(class_names)), 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(DARK_BG)
    _base_style(ax)

    for vals, offset, label, color in [
        (prec, -width, 'Precision', '#89dceb'),
        (rec,  0,      'Recall',    '#a6e3a1'),
        (f1,   width,  'F1 Score',  '#fab387'),
    ]:
        bars = ax.bar(x + offset, vals, width, label=label, color=color, edgecolor=DARK_BG)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=8, color=TEXT_COLOR)

    macro_f1 = metrics['macro_f1']
    ax.axhline(y=macro_f1, color='#f9e2af', linestyle='--', linewidth=1.5, label=f'Macro F1 = {macro_f1:.4f}')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title(f'Per-Class Metrics — {experiment_name}', fontsize=13, color=TEXT_COLOR)
    ax.legend(fontsize=9, facecolor=PANEL_BG, labelcolor=TEXT_COLOR, framealpha=0.8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
        logger.info(f'Saved: {save_path}')
    plt.show()


def plot_error_analysis(misclassified: List[Dict], max_show: int = 12, save_path=None, experiment_name='ResNet50 (Exp 01)'):
    samples = misclassified[:max_show]
    if not samples:
        logger.info('Tidak ada sampel salah klasifikasi untuk ditampilkan.')
        return

    n_cols = 4
    n_rows = (len(samples) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3.5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle(f'Error Analysis — {experiment_name}\n(Gambar Salah Diklasifikasikan)',
                 fontsize=13, fontweight='bold', color=TEXT_COLOR)

    axes_flat = np.array(axes).flatten()
    for ax, sample in zip(axes_flat, samples):
        try:
            img = Image.open(sample['filepath']).convert('RGB')
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'Load Error', ha='center', va='center', color=TEXT_COLOR, transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_facecolor(DARK_BG)
        for spine in ax.spines.values():
            spine.set_edgecolor(ACCENT)
            spine.set_linewidth(2)
            spine.set_visible(True)
        ax.set_title(f"True: {sample['true_class']}\nPred: {sample['pred_class']}",
                     fontsize=8, color=TEXT_COLOR, pad=4)
    for ax in axes_flat[len(samples):]:
        ax.set_visible(False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
        logger.info(f'Saved: {save_path}')
    plt.show()


def save_metrics(metrics: Dict, output_dir: Path, filename: str) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    with open(path, 'w') as f:
        json.dump(metrics, f, indent=2)
    logger.info(f'Metrics saved: {path}')


print('✅ Semua fungsi Evaluasi & Visualisasi didefinisikan.')

In [ ]:
# ============================================================
# EVALUATION STEP 1: Load Best Model Checkpoint
# ============================================================
BEST_MODEL_PATH = CHECKPOINTS_DIR / f'{EXP_NAME}_best_model.pth'

if BEST_MODEL_PATH.exists():
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    print(f'✅ Best model dimuat dari epoch {checkpoint["epoch"]}.')
else:
    print('⚠️  Checkpoint tidak ditemukan. Menggunakan model akhir training.')

model = model.to(DEVICE)

In [ ]:
# ============================================================
# EVALUATION STEP 2: Run Inference on Validation Set
# ============================================================
print('🔍 Menjalankan inferensi pada validation set...')
y_pred, y_true, y_probs = run_inference(model, val_loader, DEVICE)

print(f'✅ Inference selesai.')
print(f'   Total Sampel: {len(y_true):,}')
print(f'   Distribusi true labels: {dict(zip(*np.unique(y_true, return_counts=True)))}')

In [ ]:
# ============================================================
# EVALUATION STEP 3: Compute Metrics & Classification Report
# ============================================================
metrics = compute_metrics(y_true, y_pred, class_names=CLASS_NAMES)

print('=' * 56)
print('  📊 EVALUATION METRICS — ResNet50 Exp 01')
print('=' * 56)
print(f'  🏆 Macro F1 Score : {metrics["macro_f1"]:.4f}  ← PRIMARY BDC METRIC')
print(f'  🎯 Accuracy       : {metrics["accuracy"]:.4f}')
print(f'  🔍 Precision (M)  : {metrics["precision"]:.4f}')
print(f'  📦 Recall (M)     : {metrics["recall"]:.4f}')
print('=' * 56)
print()
print('📋 CLASSIFICATION REPORT:')
print(get_classification_report(y_true, y_pred, class_names=CLASS_NAMES))

# Save metrics
save_metrics(metrics, REPORTS_DIR, f'{EXP_NAME}_evaluation_metrics.json')
print(f'💾 Metrics disimpan: {REPORTS_DIR / (EXP_NAME + "_evaluation_metrics.json")}')

In [ ]:
# ============================================================
# EVALUATION STEP 4: Confusion Matrix
# ============================================================
cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(
    cm=cm,
    class_names=CLASS_NAMES,
    save_path=FIGURES_DIR / f'{EXP_NAME}_confusion_matrix.png',
    experiment_name='ResNet50 (Exp 01)',
)

In [ ]:
# ============================================================
# EVALUATION STEP 5: Per-Class Metrics Bar Chart
# ============================================================
plot_per_class_metrics(
    metrics=metrics,
    class_names=CLASS_NAMES,
    save_path=FIGURES_DIR / f'{EXP_NAME}_per_class_metrics.png',
    experiment_name='ResNet50 (Exp 01)',
)

In [ ]:
# ============================================================
# EVALUATION STEP 6: Learning Curves
# ============================================================
if 'df_history' in dir() and len(df_history) > 0:
    best_epoch = int(df_history['val_macro_f1'].idxmax()) + 1
    plot_learning_curves(
        history_df=df_history,
        best_epoch=best_epoch,
        save_path=FIGURES_DIR / f'{EXP_NAME}_learning_curves.png',
        experiment_name='ResNet50 (Exp 01)',
    )
else:
    print('⚠️  df_history tidak ditemukan. Jalankan Stage 6 terlebih dahulu.')

In [ ]:
# ============================================================
# EVALUATION STEP 7: Error Analysis (Misclassified Samples)
# ============================================================
# Ambil filepath dari validation dataset
val_dataset   = val_loader.dataset
val_filepaths = val_dataset.df['filepath'].tolist()

misclassified = get_misclassified_samples(
    y_true=y_true, y_pred=y_pred,
    filepaths=val_filepaths, class_names=CLASS_NAMES, max_samples=12,
)

print(f'🔎 Total sampel salah prediksi: {int(np.sum(y_true != y_pred)):,} dari {len(y_true):,}')
print(f'   Menampilkan {len(misclassified)} sampel error...')

plot_error_analysis(
    misclassified=misclassified,
    save_path=FIGURES_DIR / f'{EXP_NAME}_error_analysis.png',
    experiment_name='ResNet50 (Exp 01)',
)

---
## 📝 Stage 8 — Kesimpulan & Analisis

> 📌 **Instruksi:** Jalankan semua sel di atas, lalu lengkapi bagian analisis di bawah ini berdasarkan metrik dan visualisasi yang dihasilkan.

---

### 1. Interpretasi Metrik Utama

| Metrik | Nilai |
|---|---|
| **🏆 Macro F1 Score** *(BDC Primary)* | *[isi dari output sel evaluasi]* |
| **🎯 Accuracy** | *[isi]* |
| **🔍 Precision (Macro)** | *[isi]* |
| **📦 Recall (Macro)** | *[isi]* |

- **Kelas Tersulit:** Berdasarkan *Confusion Matrix*, kelas yang paling sering tertukar adalah [...] dengan [...]. Kemungkinan alasannya adalah [...]

---

### 2. Diagnosis Kurva Pembelajaran

- **Convergence:** Kurva Training dan Validation menunjukkan [Healthy Convergence / Overfitting / Underfitting].
- **Stabilitas:** [Tulis observasi tentang fluktuasi kurva...]
- **Best Epoch:** Model terbaik ditemukan di epoch [...]

---

### 3. Perbandingan dengan CNN Baseline

| Metrik | CNN Baseline | ResNet50 (Exp 01) | Peningkatan |
|---|---|---|---|
| **Macro F1** | *[isi]* | *[isi]* | *[isi]* |
| **Accuracy** | *[isi]* | *[isi]* | *[isi]* |

**Analisis:** Transfer Learning ResNet50 memberikan performa [Lebih Baik / Lebih Buruk / Sama] dibandingkan CNN Baseline, karena [alasan arsitektural / ekstraksi fitur pretrained].

---

### 4. Kesimpulan & Rekomendasi

- **Kelebihan ResNet50 pada kasus ini:** [...]
- **Kelemahan / Sisa Error:** [...]
- **Rekomendasi Selanjutnya:** [Apakah pantas lanjut ke Eksperimen 2 (Fine-Tuning)? Apakah perlu mengubah arsitektur head?]

---

### 5. Output Artefak yang Dihasilkan

| File | Lokasi | Keterangan |
|---|---|---|
| `resnet50_exp01_best_model.pth` | `outputs/checkpoints/` | Best model weights |
| `resnet50_exp01_training_history.csv` | `outputs/reports/` | Training history per epoch |
| `resnet50_exp01_evaluation_metrics.json` | `outputs/reports/` | Metrik evaluasi lengkap |
| `resnet50_exp01_confusion_matrix.png` | `outputs/figures/` | Confusion matrix |
| `resnet50_exp01_per_class_metrics.png` | `outputs/figures/` | Per-class bar chart |
| `resnet50_exp01_learning_curves.png` | `outputs/figures/` | Learning curves |
| `resnet50_exp01_error_analysis.png` | `outputs/figures/` | Error analysis grid |
| `preprocessing_split.csv` | `outputs/reports/` | Dataset split report |

In [ ]:
# ============================================================
# RINGKASAN AKHIR
# ============================================================
print('\n' + '=' * 60)
print('  ✅ EKSPERIMEN SELESAI: ResNet50 Experiment 01')
print('=' * 60)

if 'metrics' in dir():
    print(f'  🏆 HASIL AKHIR:')
    print(f'     Macro F1 Score : {metrics["macro_f1"]:.4f}  ← BDC PRIMARY METRIC')
    print(f'     Accuracy       : {metrics["accuracy"]:.4f}')
    print(f'     Precision      : {metrics["precision"]:.4f}')
    print(f'     Recall         : {metrics["recall"]:.4f}')

if 'df_history' in dir() and len(df_history) > 0:
    best_idx = df_history['val_macro_f1'].idxmax()
    print(f'\n  📈 TRAINING INFO:')
    print(f'     Total Epochs   : {len(df_history)}')
    print(f'     Best Epoch     : {int(df_history.loc[best_idx, "epoch"])}')
    print(f'     Best Val F1    : {df_history["val_macro_f1"].max():.4f}')
    print(f'     Final LR       : {df_history["lr"].iloc[-1]:.6f}')

print(f'\n  📁 OUTPUT ARTEFAK:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f'     {path.relative_to(OUTPUT_DIR)} ({size_kb:.1f} KB)')

print('=' * 60)